# Sekhmet playground

This notebook lives in the gitignored `playground/`, so edit freely. Imports
resolve to the **checked-out source** of each workspace package, so whatever
branch the repo is on is what you are playing with.

Run a cell with Shift+Enter. The Deno kernel shows the last expression's value.

## `@sekhmet/enum`: `option` and `result`

In [ ]:
import { None, type Option, Some } from "@sekhmet/enum/option";
import { Err, Ok, type Result } from "@sekhmet/enum/result";

const half = (n: number): Option<number> => (n % 2 === 0 ? Some(n / 2) : None());
const parse = (text: string): Result<number, string> =>
	Number.isNaN(Number(text)) ? Err(`not a number: ${text}`) : Ok(Number(text));

[
	Some(84).andThen(half).map((n) => n.toString()).unwrapOr("odd"),
	parse("21").map((n) => n * 2).ok().unwrapOr(0),
	parse("abc").mapErr((e) => e.toUpperCase()).unwrapErr(),
]

## `@sekhmet/enum`: `ordering`, `poll`, `latch` and `core`

In [ ]:
import { compareNumbers, compareStrings } from "@sekhmet/enum/ordering";
import { Pending, Ready } from "@sekhmet/enum/poll";
import { Active, type Latch } from "@sekhmet/enum/latch";
import { Enum, type Split } from "@sekhmet/enum/core";

const people = [
	{ name: "Bea", age: 30 },
	{ name: "Al", age: 30 },
	{ name: "Cy", age: 25 },
];
people.sort((a, b) =>
	compareNumbers(a.age, b.age).andThen(compareStrings(a.name, b.name)).toNumber(),
);

const connection: Latch<string, string> = Active("socket");
connection.deactivate("closed by peer");

type Shape = Split<{ Circle: [radius: number]; Square: [side: number] }>;
const area = new Enum<Shape>("Square", 3).match({
	Circle: (radius) => Math.PI * radius ** 2,
	Square: (side) => side ** 2,
});

({
	sorted: people.map((p) => p.name),
	ready: Ready(2).map((n) => n + 1).isReady(),
	pending: Pending<number>().map((n) => n + 1).isReady(),
	latched: connection.inactive().unwrap(),
	area,
})

## `@sekhmet/effect`

In [ ]:
import { fork, Runtime, Scheduler, Success, sync } from "@sekhmet/effect";

const effect = Success(20)
	.map((n) => n + 1)
	.andThen((n) => Success(n * 2));

const runtime = new Runtime({
	input: undefined,
	config: { scheduler: new Scheduler({ schedule: sync() }) },
});

const results: unknown[] = [];
fork(runtime, effect).join((result) => results.push(result));
results

## `@sekhmet/vow`

In [ ]:
import { map, vowWithResolver } from "@sekhmet/vow";

const { vow, resolve } = vowWithResolver<number>();
const seen: number[] = [];
map(vow, (n) => seen.push(n * 2));

resolve(21); // dependents run synchronously, inside resolve
seen

## `@sekhmet/panic`

In [ ]:
import { definePanicRuntime, Panic, panicWithoutCause } from "@sekhmet/panic";

const reports: string[] = [];
const panics = definePanicRuntime();
panics.setHook(({ payload }) => {
	reports.push(payload instanceof Panic ? payload.message : String(payload));
});

const answer = panics.catchUnwind(() => 42);
panics.catchUnwind(() => panicWithoutCause({ message: "broken invariant" }));
({ answer, reports })

## `@sekhmet/matrix` and `@sekhmet/primordials`

In [ ]:
import { matrix } from "@sekhmet/matrix";
import { getStructParams, Struct } from "@sekhmet/primordials/struct";
import { block } from "@sekhmet/primordials/scoping";

class Point extends Struct<{ x: number; y: number }> {}

({
	cases: matrix({ size: [1, 2], unit: ["px", "em"] }),
	x: getStructParams(new Point({ x: 1, y: 2 })).x,
	sum: block(() => 1 + 2),
})